# FaultSense — CWRU Bearing Dataset, End-to-End Training

**Course:** LLMs: A Hands-on Approach, CCE IISc  
**Dataset:** Case Western Reserve University (CWRU) Bearing Fault Dataset  
**Model:** Qwen2.5-7B-Instruct, 4-bit QLoRA  

This notebook uses CWRU bearing data, which has **real, lab-verified fault-type labels**
(Inner Race, Outer Race, Ball, Normal) at three severity levels (Low/Medium/High).

### Pipeline
1. Download the CWRU dataset from Kaggle (`brjapon/cwru-bearing-datasets`).
2. Load the pre-extracted time-domain feature CSV — no raw signal processing needed.
3. Map the raw fault labels to clean `"Fault Type – Severity"` strings.
4. Serialise into instruction-tuning JSONL.
5. Split into train/val/test (80/10/10).
6. QLoRA fine-tune Qwen2.5-7B on a T4 GPU.
7. Evaluate on held-out test set.

### Labels (10 classes, 230 examples each)
| Fault Type | Severity | Fault Diameter |
|---|---|---|
| Normal | Healthy | No fault |
| Inner Race | Low | 0.007" |
| Inner Race | Medium | 0.014" |
| Inner Race | High | 0.021" |
| Outer Race | Low | 0.007" |
| Outer Race | Medium | 0.014" |
| Outer Race | High | 0.021" |
| Ball | Low | 0.007" |
| Ball | Medium | 0.014" |
| Ball | High | 0.021" |

---
# Section 1 — Environment Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

project_path = "/content/drive/MyDrive/FaultSense_Project"
os.makedirs(project_path, exist_ok=True)

for folder in ["data", "finetune", "rag", "evaluation"]:
    os.makedirs(folder, exist_ok=True)

print("Project folder:", project_path)
print("Local folders:", os.listdir("."))

Project folder: /content/drive/MyDrive/FaultSense_Project
Local folders: ['.config', 'finetune', 'drive', 'rag', 'evaluation', 'data', 'sample_data']


In [3]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("PyTorch:", torch.__version__)
print("bf16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

CUDA available: True
GPU: Tesla T4
PyTorch: 2.10.0+cu128
bf16 supported: True


---
# Section 2 — Download CWRU Dataset from Kaggle

We use the `brjapon/cwru-bearing-datasets` dataset. It contains pre-extracted
time-domain features in a CSV.

**Steps to get your Kaggle API token:**
1. Go to https://www.kaggle.com/settings → "Create New API Token" → downloads `kaggle.json`
2. Run the upload cell below and select that file.

In [4]:
!pip install -q kaggle scipy

In [5]:
from google.colab import files
files.upload()  # select your kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"namitab_cce_llm","key":"01f41290647788e791d9eb7498689e6e"}'}

In [6]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
print("kaggle.json installed.")

kaggle.json installed.


In [7]:
KAGGLE_DATASET = "brjapon/cwru-bearing-datasets"

if not os.path.exists("data/feature_time_48k_2048_load_1.csv"):
    !kaggle datasets download -d {KAGGLE_DATASET} --path data/ --unzip
    print("Download complete. Contents of data/:")
    print(os.listdir("data"))
else:
    print("Dataset already exists, skipping download.")

Dataset URL: https://www.kaggle.com/datasets/brjapon/cwru-bearing-datasets
License(s): CC-BY-SA-4.0
100% 40.4M/40.4M [00:00<00:00, 97.3MB/s]

Download complete. Contents of data/:
['raw', 'feature_time_48k_2048_load_1.csv', 'CWRU_48k_load_1_CNN_data.npz']


---
# Section 3 — Load the Pre-Extracted Feature CSV

The dataset already contains time-domain vibration features computed from
2048-sample windows at 48kHz.

**9 features per window:**

| Column | What it measures |
|---|---|
| `max` | Maximum amplitude |
| `min` | Minimum amplitude |
| `mean` | Average value |
| `sd` | Standard deviation |
| `rms` | Root mean square — rises as damage worsens |
| `skewness` | Asymmetry of the vibration waveform |
| `kurtosis` | Spike sharpness — healthy ≈ 0, faulty >> 0 |
| `crest` | Peak ÷ RMS — high means sharp impacts |
| `form` | RMS ÷ mean(|x|) — waveform shape |

**10 classes, 230 examples each (perfectly balanced):**

| Raw label | Our label |
|---|---|
| `Ball_007_1` | `Ball – Low` |
| `Ball_014_1` | `Ball – Medium` |
| `Ball_021_1` | `Ball – High` |
| `IR_007_1` | `Inner Race – Low` |
| `IR_014_1` | `Inner Race – Medium` |
| `IR_021_1` | `Inner Race – High` |
| `OR_007_6_1` | `Outer Race – Low` |
| `OR_014_6_1` | `Outer Race – Medium` |
| `OR_021_6_1` | `Outer Race – High` |
| `Normal_1` | `Normal – Healthy` |

In [8]:
import pandas as pd

df = pd.read_csv("data/feature_time_48k_2048_load_1.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print()
print(df.head())
print()
print("Class distribution:")
print(df['fault'].value_counts())

Shape: (2300, 10)
Columns: ['max', 'min', 'mean', 'sd', 'rms', 'skewness', 'kurtosis', 'crest', 'form', 'fault']

       max      min      mean        sd       rms  skewness  kurtosis  \
0  0.35986 -0.41890  0.017840  0.122746  0.124006 -0.118571 -0.042219   
1  0.46772 -0.36111  0.022255  0.132488  0.134312  0.174699 -0.081548   
2  0.46855 -0.43809  0.020470  0.149651  0.151008  0.040339 -0.274069   
3  0.58475 -0.54303  0.020960  0.157067  0.158422 -0.023266  0.134692   
4  0.44685 -0.57891  0.022167  0.138189  0.139922 -0.081534  0.402783   

      crest      form       fault  
0  2.901946  6.950855  Ball_007_1  
1  3.482334  6.035202  Ball_007_1  
2  3.102819  7.376926  Ball_007_1  
3  3.691097  7.558387  Ball_007_1  
4  3.193561  6.312085  Ball_007_1  

Class distribution:
fault
Ball_007_1    230
Ball_014_1    230
Ball_021_1    230
IR_007_1      230
IR_014_1      230
IR_021_1      230
OR_007_6_1    230
OR_014_6_1    230
OR_021_6_1    230
Normal_1      230
Name: count, dtype: int6

In [9]:
LABEL_MAP = {
    'Ball_007_1':  'Ball – Low',
    'Ball_014_1':  'Ball – Medium',
    'Ball_021_1':  'Ball – High',
    'IR_007_1':    'Inner Race – Low',
    'IR_014_1':    'Inner Race – Medium',
    'IR_021_1':    'Inner Race – High',
    'OR_007_6_1':  'Outer Race – Low',
    'OR_014_6_1':  'Outer Race – Medium',
    'OR_021_6_1':  'Outer Race – High',
    'Normal_1':    'Normal – Healthy',
}

df['label'] = df['fault'].map(LABEL_MAP)

# Verify every raw label was mapped — unmapped rows would show NaN
unmapped = df[df['label'].isna()]
if len(unmapped) > 0:
    print("WARNING — unmapped labels:", unmapped['fault'].unique())
else:
    print("All labels mapped successfully.")

print()
print(df[['fault', 'label']].drop_duplicates().to_string(index=False))

All labels mapped successfully.

     fault               label
Ball_007_1          Ball – Low
Ball_014_1       Ball – Medium
Ball_021_1         Ball – High
  IR_007_1    Inner Race – Low
  IR_014_1 Inner Race – Medium
  IR_021_1   Inner Race – High
OR_007_6_1    Outer Race – Low
OR_014_6_1 Outer Race – Medium
OR_021_6_1   Outer Race – High
  Normal_1    Normal – Healthy


---
# Section 4 — Build the Instruction-Tuning JSONL

Each row in the CSV becomes one training example in Alpaca format:

```
{
  "instruction": "Analyze bearing vibration features and identify fault type and severity",
  "input": "Max=0.3599, Min=-0.4189, Mean=0.0178, StdDev=0.1227, RMS=0.1240,
            Skewness=-0.1186, Kurtosis=-0.0422, CrestFactor=2.9019, FormFactor=6.9509",
  "output": "Ball – Low"
}
```

In [10]:
import json

INSTRUCTION = "Analyze bearing vibration features and identify fault type and severity"

FEATURE_COLS = {
    'max':      'Max',
    'min':      'Min',
    'mean':     'Mean',
    'sd':       'StdDev',
    'rms':      'RMS',
    'skewness': 'Skewness',
    'kurtosis': 'Kurtosis',
    'crest':    'CrestFactor',
    'form':     'FormFactor',
}

all_examples = []
for _, row in df.iterrows():
    input_str = ", ".join(
        f"{friendly}={row[col]:.4f}"
        for col, friendly in FEATURE_COLS.items()
    )
    all_examples.append({
        "instruction": INSTRUCTION,
        "input":       input_str,
        "output":      row['label'],
    })

with open("train_cwru.json", "w") as f:
    for ex in all_examples:
        f.write(json.dumps(ex) + "\n")

print(f"Wrote {len(all_examples)} examples to train_cwru.json")
print()
print("Sample:")
print("INPUT: ", all_examples[0]["input"])
print("OUTPUT:", all_examples[0]["output"])

Wrote 2300 examples to train_cwru.json

Sample:
INPUT:  Max=0.3599, Min=-0.4189, Mean=0.0178, StdDev=0.1227, RMS=0.1240, Skewness=-0.1186, Kurtosis=-0.0422, CrestFactor=2.9019, FormFactor=6.9509
OUTPUT: Ball – Low


---
# Section 5 — Train / Validation / Test Split + Data Augmentation

Split: **80% train / 10% val / 10% test** (1840 / 230 / 230)

### Why augment?

The CWRU dataset has only **2,300 examples total** — 230 per class. After the 80/10/10 split,
the train set has just **1,840 examples**. This is too small for reliable LLM fine-tuning:

- General SFT guidelines recommend **5k–50k examples** for stable generalisation
- With only 230 examples per class, the model sees very little feature variation per fault type
- Risk of **overfitting to the template** rather than learning the underlying fault patterns

### What augmentation does

Augmentation is handled by the external script **`augment_cwru.py`** which reads
`train_split.json` and writes `augmented_train.json`. Two techniques are applied
to the train split only:

1. **Instruction phrasing diversity** — each example is reworded with a different question
   (e.g. *"Classify the fault"* → *"Diagnose the bearing condition"* → *"What fault is present?"*)
   so the model learns fault semantics, not just one fixed prompt format

2. **Gaussian feature noise** — small random perturbation (2% of each feature's std) is added
   to simulate natural sensor variation across measurement windows

This grows the train set from **1,840 → 7,360 examples (4×)** while keeping val and test
as original, unaugmented examples for clean evaluation.

In [11]:
from datasets import load_dataset

raw = load_dataset("json", data_files="train_cwru.json")["train"]
print(f"Total samples: {len(raw)}")

split1 = raw.train_test_split(test_size=0.2, seed=42)
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)

train_ds_raw = split1["train"]   # 1840 — will be augmented
val_ds_raw   = split2["train"]   # 230  — original, untouched
test_ds_raw  = split2["test"]    # 230  — original, untouched

print(f"Train: {len(train_ds_raw)} | Val: {len(val_ds_raw)} | Test: {len(test_ds_raw)}")

# Save train split and test set to disk
train_ds_raw.to_json("train_split.json")
test_ds_raw.to_json("test_cwru.json")
print("Saved train_split.json and test_cwru.json")

# Run augmentation only if not already done
if not os.path.exists("augmented_train.json"):
    !python augment_cwru.py
else:
    print("augmented_train.json already exists, skipping augmentation.")

# Load augmented train for fine-tuning
train_ds_raw = load_dataset("json", data_files="augmented_train.json")["train"]
print(f"\nFinal sizes  →  Train: {len(train_ds_raw)} | Val: {len(val_ds_raw)} | Test: {len(test_ds_raw)}")

Generating train split: 0 examples [00:00, ? examples/s]

Total samples: 2300
Train: 1840 | Val: 230 | Test: 230


Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved train_split.json and test_cwru.json
Loaded 1840 examples from train_split.json
Augmented examples: 7360  (4x)
Saved to: augmented_train.json

Class distribution:
  Ball – High                  748
  Ball – Low                   748
  Ball – Medium                744
  Inner Race – High            744
  Inner Race – Low             716
  Inner Race – Medium          716
  Normal – Healthy             752
  Outer Race – High            736
  Outer Race – Low             732
  Outer Race – Medium          724


Generating train split: 0 examples [00:00, ? examples/s]


Final sizes  →  Train: 7360 | Val: 230 | Test: 230


In [12]:
def format_example(example):
    return {
        "text": (
            "### Instruction:\n"
            f"{example['instruction']}\n\n"
            "### Input:\n"
            f"{example['input']}\n\n"
            "### Response:\n"
            f"{example['output']}"
        )
    }

train_ds = train_ds_raw.map(format_example)
val_ds   = val_ds_raw.map(format_example)
test_ds  = test_ds_raw.map(format_example)

# Persist test set to disk for reuse
test_ds_raw.to_json("test_cwru.json")
print("Test set saved to test_cwru.json")
print()
print("Sample formatted training example:")
print(train_ds[0]["text"])

Map:   0%|          | 0/7360 [00:00<?, ? examples/s]

Map:   0%|          | 0/230 [00:00<?, ? examples/s]

Map:   0%|          | 0/230 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Test set saved to test_cwru.json

Sample formatted training example:
### Instruction:
Analyze bearing vibration features and identify fault type and severity

### Input:
Max=4.8540, Min=-4.7568, Mean=0.0141, StdDev=0.9650, RMS=0.9648, Skewness=0.0357, Kurtosis=4.6378, CrestFactor=5.0310, FormFactor=68.4093

### Response:
Outer Race – Low


---
# Section 6 — Install Training Libraries

In [13]:
!pip install -q "transformers>=4.44,<4.50" "peft>=0.11,<0.13" "trl>=0.9,<0.12" \
    "accelerate>=0.33" "bitsandbytes>=0.43" "datasets>=2.20" sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 19.1 MB/s eta 0:00:00


---
# Section 7 — Load Qwen2.5-7B in 4-bit (QLoRA)

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training
import torch

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Pad token:", tokenizer.pad_token, "| ID:", tokenizer.pad_token_id)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

model = prepare_model_for_kbit_training(model)
print("Model loaded and prepared for kbit training.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  # Means the user did not define a `HF_TOKEN` secret => warn


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded and prepared for kbit training.


---
# Section 8 — LoRA Configuration

We target **all 7 linear layers** — attention + MLP.

`r=8, alpha=16` targeting all 7 layers. With ~18,400 augmented training examples
(10× the original), there is enough data to fine-tune both attention and MLP layers
without high risk of memorisation.

To run the r=16 ablation, change `r=8` → `r=16` and re-run from this cell.

In [15]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention layers
        "gate_proj", "up_proj", "down_proj",        # MLP layers
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expect ~20M trainable / 7.6B total (~0.26%)

trainable params: 20,185,088 || all params: 7,635,801,600 || trainable%: 0.2643


---
# Section 9 — Training Arguments (T4-safe)

In [16]:
# ── Experiment identifier — change this for each run ─────────────────────────
# Format: r{lora_r}_{layers}_{epochs}ep
# Examples: "r8_all7_2ep", "r16_all7_2ep", "r8_attn_2ep"

RUN_TAG = "r8_all7_2ep"

output_dir     = f"./finetune/{RUN_TAG}"
drive_save_dir = f"...FaultSense_Project/finetune/{RUN_TAG}"
os.makedirs(drive_save_dir, exist_ok=True)

print(f"Run tag      : {RUN_TAG}")
print(f"Output dir   : {output_dir}")
print(f"Drive save   : {drive_save_dir}")

Run tag      : r8_all7_2ep
Output dir   : ./qwen-cwru-lora-r8_all7_2ep
Drive save   : /content/drive/MyDrive/FaultSense_Project/r8_all7_2ep


In [18]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    report_to="none",
)
print("TrainingArguments configured.")

TrainingArguments configured.


---
# Section 10 — SFTTrainer

In [19]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=256,      # prompts are short (~60 tokens), 256 is generous
    packing=False,
)
print("SFTTrainer ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/7360 [00:00<?, ? examples/s]

Map:   0%|          | 0/230 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


SFTTrainer ready.


---
# Section 11 — Train

2 epochs on a T4 GPU. Expect approx 3 hrs training time.

In [20]:
trainer.train()

Step,Training Loss,Validation Loss
50,0.717700,0.715295
100,0.678600,0.651773
150,0.658500,0.635766
200,0.666100,0.625121
250,0.653300,0.611148
300,0.638500,0.603563
350,0.635000,0.599776
400,0.640400,0.600525
450,0.636900,0.588941
500,0.633300,0.591716


TrainOutput(global_step=1840, training_loss=0.6509648525196573, metrics={'train_runtime': 11195.1668, 'train_samples_per_second': 1.315, 'train_steps_per_second': 0.164, 'total_flos': 7.237827795102106e+16, 'train_loss': 0.6509648525196573, 'epoch': 2.0})

---
# Section 14 — Save the LoRA Adapter

Experiment identifier: change this for each run
Format: r{lora_r}_{layers}_{epochs}ep

Examples: "r8_all7_2ep", "r16_all7_2ep", "r8_attn_2ep"


In [23]:
RUN_TAG = "r8_all7_2ep"

output_dir     = f"./finetune/{RUN_TAG}"
drive_save_dir = f"/content/drive/MyDrive/FaultSense_Project/finetune/{RUN_TAG}"
os.makedirs(drive_save_dir, exist_ok=True)

print(f"Run tag      : {RUN_TAG}")
print(f"Output dir   : {output_dir}")
print(f"Drive save   : {drive_save_dir}")

Run tag      : r8_all7_2ep
Output dir   : ./finetune/r8_all7_2ep
Drive save   : /content/drive/MyDrive/FaultSense_Project/finetune/r8_all7_2ep


In [24]:
import shutil

trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Adapter saved locally to {output_dir}")

if os.path.exists("/content/drive/MyDrive"):
    shutil.copytree(output_dir, drive_save_dir, dirs_exist_ok=True)
    shutil.copy("test_cwru.json",  f"{drive_save_dir}/test_cwru.json")
    shutil.copy("train_cwru.json", f"{drive_save_dir}/train_cwru.json")
    print(f"Adapter and datasets copied to Drive: {drive_save_dir}")

Adapter saved locally to ./finetune/r8_all7_2ep
Adapter and datasets copied to Drive: /content/drive/MyDrive/FaultSense_Project/finetune/r8_all7_2ep


---
# Section 15 — Evaluation



In [25]:
import torch

model.eval()

VALID_LABELS = set(LABEL_MAP.values())

def predict(instruction: str, sensor_input: str, max_new_tokens: int = 16) -> str:
    prompt = (
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Input:\n"
        f"{sensor_input}\n\n"
        "### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            stop_strings=["\n", "###"],   # stop at newline or next section
            tokenizer=tokenizer,
        )
    raw = tokenizer.decode(
        out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    ).strip()

    # Extract first valid label even if model runs multiple labels on one line
    for label in VALID_LABELS:
        if raw.startswith(label):
            return label

    return raw.split("\n")[0].strip()   # fallback for debugging


print("=" * 70)
print("SPOT CHECK — 5 held-out test examples")
print("=" * 70)
for i in range(5):
    ex   = test_ds_raw[i]
    pred = predict(ex["instruction"], ex["input"])
    match = "PASS" if pred == ex["output"] else "FAIL"
    print(f"\n[{i}] {match}")
    print(f"  Input:     {ex['input']}")
    print(f"  Expected:  {ex['output']}")
    print(f"  Predicted: {pred}")

SPOT CHECK — 5 held-out test examples


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



[0] PASS
  Input:     Max=3.9313, Min=-4.1508, Mean=0.0148, StdDev=0.6629, RMS=0.6629, Skewness=-0.2887, Kurtosis=8.6678, CrestFactor=5.9304, FormFactor=44.7884
  Expected:  Outer Race – High
  Predicted: Outer Race – High

[1] PASS
  Input:     Max=0.7965, Min=-0.6905, Mean=0.0313, StdDev=0.1835, RMS=0.1861, Skewness=0.0199, Kurtosis=0.8290, CrestFactor=4.2791, FormFactor=5.9470
  Expected:  Inner Race – Medium
  Predicted: Inner Race – Medium

[2] PASS
  Input:     Max=0.4218, Min=-0.4540, Mean=0.0141, StdDev=0.1407, RMS=0.1413, Skewness=-0.0626, Kurtosis=-0.0831, CrestFactor=2.9848, FormFactor=10.0355
  Expected:  Outer Race – Medium
  Predicted: Outer Race – Medium

[3] PASS
  Input:     Max=3.4589, Min=-3.6600, Mean=0.0166, StdDev=0.6479, RMS=0.6480, Skewness=-0.2318, Kurtosis=7.5686, CrestFactor=5.3382, FormFactor=39.0783
  Expected:  Outer Race – High
  Predicted: Outer Race – High

[4] PASS
  Input:     Max=1.3627, Min=-1.4995, Mean=0.0194, StdDev=0.2789, RMS=0.2795, Skewness=

In [26]:
from collections import defaultdict

n = min(200, len(test_ds_raw))
correct = 0
per_class_correct = defaultdict(int)
per_class_total   = defaultdict(int)

for i in range(n):
    ex       = test_ds_raw[i]
    pred     = predict(ex["instruction"], ex["input"])
    expected = ex["output"]

    per_class_total[expected] += 1
    if pred == expected:          # exact match
        correct += 1
        per_class_correct[expected] += 1

print(f"Overall accuracy on {n} test samples: {correct}/{n} = {correct/n:.1%}")
print()
print("Per-class accuracy:")
print(f"  {'Class':<28} {'Correct':>8} {'Total':>8} {'Acc':>8}")
print("  " + "-" * 56)
for cls in sorted(per_class_total.keys()):
    tot = per_class_total[cls]
    cor = per_class_correct[cls]
    print(f"  {cls:<28} {cor:>8} {tot:>8} {cor/tot:>7.1%}")

Overall accuracy on 200 test samples: 185/200 = 92.5%

Per-class accuracy:
  Class                         Correct    Total      Acc
  --------------------------------------------------------
  Ball – High                        15       19   78.9%
  Ball – Low                         14       15   93.3%
  Ball – Medium                      17       20   85.0%
  Inner Race – High                  16       16  100.0%
  Inner Race – Low                   15       15  100.0%
  Inner Race – Medium                22       22  100.0%
  Normal – Healthy                   23       23  100.0%
  Outer Race – High                  24       25   96.0%
  Outer Race – Low                   20       21   95.2%
  Outer Race – Medium                19       24   79.2%


In [28]:
import json, datetime

results = {
    "run_tag":          RUN_TAG,
    "timestamp":        datetime.datetime.now().isoformat(),
    "lora_r":           lora_config.r,
    "lora_alpha":       lora_config.lora_alpha,
    "target_modules":   list(lora_config.target_modules),
    "epochs":           training_args.num_train_epochs,
    "train_size":       len(train_ds_raw),
    "overall_accuracy": round(correct / n, 4),
    "per_class_accuracy": {
        cls: round(per_class_correct[cls] / per_class_total[cls], 4)
        for cls in sorted(per_class_total)
    },
}

results_path = f"{drive_save_dir}/results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {results_path}")
print(json.dumps(results, indent=2))

Results saved to /content/drive/MyDrive/FaultSense_Project/finetune/r8_all7_2ep/results.json
{
  "run_tag": "r8_all7_2ep",
  "timestamp": "2026-05-01T01:46:03.989024",
  "lora_r": 8,
  "lora_alpha": 16,
  "target_modules": [
    "up_proj",
    "q_proj",
    "v_proj",
    "gate_proj",
    "o_proj",
    "k_proj",
    "down_proj"
  ],
  "epochs": 2,
  "train_size": 7360,
  "overall_accuracy": 0.925,
  "per_class_accuracy": {
    "Ball \u2013 High": 0.7895,
    "Ball \u2013 Low": 0.9333,
    "Ball \u2013 Medium": 0.85,
    "Inner Race \u2013 High": 1.0,
    "Inner Race \u2013 Low": 1.0,
    "Inner Race \u2013 Medium": 1.0,
    "Normal \u2013 Healthy": 1.0,
    "Outer Race \u2013 High": 0.96,
    "Outer Race \u2013 Low": 0.9524,
    "Outer Race \u2013 Medium": 0.7917
  }
}


---
## Run: `r8_all7_2ep` — Results Summary

| | |
|---|---|
| LoRA rank / alpha | r=8, alpha=16 |
| Target layers | All 7 (q/k/v/o + gate/up/down) |
| Trainable params | 20.18M (0.26%) |
| Train examples | 7,360 (4× augmented) |
| Epochs | 2 |
| Total steps | 1,840 |
| Training loss | 0.6510 |
| Training time | ~3 hrs 6 min (T4) |
| **Overall accuracy** | **92.5%** (185/200) |

**Per-class accuracy:**

| Class | Acc |
|---|---|
| Normal – Healthy | 100% |
| Inner Race – Low | 100% |
| Inner Race – Medium | 100% |
| Inner Race – High | 100% |
| Outer Race – High | 96.0% |
| Outer Race – Low | 95.2% |
| Ball – Low | 93.3% |
| Ball – Medium | 85.0% |
| Outer Race – Medium | 79.2% |
| Ball – High | 78.9% |

**Weak classes:** Ball (all severities) and Outer Race – Medium — targets for next run.

### Training & Validation Loss Analysis

| Checkpoint | Train Loss | Val Loss |
|---|---|---|
| Step 50 (early) | 0.7177 | 0.7153 |
| Step 1840 / Epoch 2 (final) | 0.6510 | — |

**Observations:**

- **No overfitting** — at step 50, train loss (0.7177) and val loss (0.7153) are nearly identical, a gap of only 0.002. This confirms the augmented dataset is diverse enough that the model is not memorising examples.

- **Steady convergence** — loss dropped from 0.7177 → 0.6510 over 1,840 steps (~0.067 reduction). The decrease is gradual and stable, consistent with a well-regularised QLoRA run.

- **Loss did not go very low** — a final loss of 0.65 is moderate for a 10-class task. This is expected because the 4× augmentation introduces varied instruction phrasings and noisy features, making it harder for the model to memorise patterns — which is exactly what we want. The 92.5% test accuracy confirms the model generalises well despite the moderate loss.

- **2 epochs was sufficient** — the loss was still decreasing at the end of epoch 2, so a 3rd epoch could lower it further, but risks overfitting to augmentation patterns. This is a trade-off to explore in the next run.

### Accuracy Analysis

**Overall: 92.5%** (185/200 test examples, exact match)

#### Class groups

**Perfect (100%):** Normal – Healthy, Inner Race – Low/Medium/High
- These classes have the most distinct feature signatures. Inner race faults produce sharp, periodic impulses clearly visible in RMS, Kurtosis, and CrestFactor. Normal bearings have uniformly low values across all features. The model learned these boundaries cleanly.

**Strong (95–96%):** Outer Race – High, Outer Race – Low
- Outer race faults at extreme severities (low and high diameter) produce well-separated feature values, making them easy to distinguish from other classes.

**Moderate (79–93%):** Ball – Low/Medium/High, Outer Race – Medium
- **Ball faults** are consistently the weakest fault family across all severities. Ball defects produce more complex, irregular vibration patterns that do not map as cleanly to the 9 time-domain features. The model struggles to separate Ball – High from Ball – Medium, likely because the RMS and Kurtosis values overlap at these severity levels.
- **Outer Race – Medium** (79.2%) is the biggest surprise — it sits between Low and High severity, and its feature values overlap with both neighbours, causing misclassification at the boundary.

#### Severity pattern
Across all fault types, **Low severity is easier to classify than Medium**, and **High is easier than Medium**. Medium severity sits at a feature overlap zone between Low and High, making it the hardest severity level to pin down for both Ball and Outer Race classes.

#### What this means for the next run
The accuracy ceiling for this feature set and dataset may be around 93–95%. To push higher:
- Increasing `r` (e.g. r=16) gives the model more capacity to learn subtle boundaries
- Adding frequency-domain features (FFT peaks, envelope spectrum) would give sharper signal for Ball faults specifically
- More real training data for the weak classes would help more than further augmentation

---
## Run: `r16_all7_2ep` - Results Summary

This run repeats the same all-7-layer LoRA setup as `r8_all7_2ep`, but increases LoRA capacity from rank 8 to rank 16.

| | |
|---|---|
| LoRA rank / alpha | r=16, alpha=32 |
| Target layers | All 7 (q/k/v/o + gate/up/down) |
| Trainable params | 40.37M (0.5273%) |
| Train examples | 7,360 (4x augmented) |
| Epochs | 2 |
| Total steps | 1,840 |
| Training time | ~3 hrs 10 min (T4) |
| Final train loss | 0.6483 |
| Best/late validation loss | ~0.5698 near final checkpoints |
| **Overall accuracy** | **92.5%** (185/200) |

**Per-class accuracy:**

| Class | r8 Acc | r16 Acc | Change |
|---|---:|---:|---:|
| Normal - Healthy | 100.0% | 100.0% | 0.0 pp |
| Inner Race - Low | 100.0% | 100.0% | 0.0 pp |
| Inner Race - Medium | 100.0% | 100.0% | 0.0 pp |
| Inner Race - High | 100.0% | 100.0% | 0.0 pp |
| Outer Race - High | 96.0% | 96.0% | 0.0 pp |
| Outer Race - Low | 95.2% | 95.2% | 0.0 pp |
| Ball - Low | 93.3% | 93.3% | 0.0 pp |
| Ball - Medium | 85.0% | 80.0% | -5.0 pp |
| Outer Race - Medium | 79.2% | 83.3% | +4.1 pp |
| Ball - High | 78.9% | 78.9% | 0.0 pp |

### Training & Validation Loss Comparison

| Run | Checkpoint | Train Loss | Val Loss |
|---|---|---:|---:|
| r8 | Step 50 | 0.7177 | 0.7153 |
| r8 | Step 1800 | 0.6152 | 0.5682 |
| r8 | Final trainer-reported loss / step 1840 | 0.6510 | - |
| r16 | Step 50 | 0.7306 | 0.7467 |
| r16 | Step 1000 | 0.6142 | 0.5784 |
| r16 | Step 1800 | 0.6134 | 0.5698 |
| r16 | Final trainer-reported loss / step 1840 | 0.6483 | - |

**Observations:**

- The r16 model has roughly double the trainable parameters of r8 (40.37M vs 20.18M), but the same overall exact-match accuracy: 92.5%.
- Both runs reached almost identical late validation loss at step 1800: r8 = 0.5682 and r16 = 0.5698.
- The similar validation losses and identical accuracy suggest that the remaining errors are likely class-boundary or feature-representation issues rather than simply insufficient LoRA capacity.

### Rank Ablation: r8 vs r16

The rank ablation shows that increasing LoRA rank from 8 to 16 did not improve overall accuracy. Instead, it redistributed errors across the weak classes:

- **Outer Race - Medium improved** from 79.2% to 83.3%.
- **Ball - Medium dropped** from 85.0% to 80.0%.
- Ball - High stayed at 78.9%, remaining the weakest class.
- Normal and all Inner Race classes stayed perfect at 100%.

**Conclusion:** r=8 is the better default for this setup because it reaches the same 92.5% accuracy with half the trainable parameters and slightly shorter training time. The r16 run confirms that the main bottleneck is probably not adapter rank. The harder classes, especially Ball faults and medium-severity boundary cases, likely need richer features such as frequency-domain or envelope-spectrum features, or more real examples, rather than only a larger LoRA adapter.



